<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/09.Asthma_Mortality_EDA_Predictor_Variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📊 Exploratory Data Analysis (EDA) of Predictor Variables

In this notebook, we will perform an exploratory data analysis (EDA) of the predictor variables to better understand their temporal and spatial dynamics and their potential association with the asthma mortality rate (AMR) in Argentina. The analysis begins by loading and preparing the data, including checks for missing values and internal consistency to ensure data quality. We will then examine the statistical behavior of each predictor through distributional analyses, temporal trends, treemaps, and spatial visualizations such as departmental choropleth maps.


We will also evaluate the relationship between the predictors and AMR through correlation analysis, interaction effects, lagged associations, and multicollinearity diagnostics. Correlation matrices, as well as dimensionality reduction techniques such as Principal Component Analysis (PCA) and clustering, will be used to identify underlying structures within the predictor space. Together, these analyses provide a foundation for subsequent modeling and inference on the environmental predictors of AMR.

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd
import numpy as np

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
import seaborn as sns
import squarify

# statistical libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# sklearn libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# geospatial libraries
import geopandas as gpd
import mapclassify

# other libraries
from datetime import datetime
from itables import init_notebook_mode, show
import os

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Pandas Copy-on-Write
 pd.options.mode.copy_on_write = True

## 💾 Load and reduce data

Our working dataset, which contains both the predictor variables and the target variable, has a long name. For simplicity, we will rename it to `data`.

In [ ]:
# Load data
gdf_old = gpd.read_file("pdt/asthma_mortality/data/gpkg/tma_pm25_pm10_ba_pd_agrt_nwvt_nbut_2001_2022.gpkg")

In [ ]:
# save the dataset as data.gpkg
gdf_old.to_file("pdt/asthma_mortality/data/gpkg/data.gpkg", driver="GPKG")

In [ ]:
# Load data
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# visualize geo data.frame
init_notebook_mode(all_interactive=True)
gdf

In [ ]:
# inspect any nan in columns
init_notebook_mode(all_interactive=True)
gdf.isna().sum()

In [ ]:
# Reshape df to ts long format
years = range(2001, 2023)
records = []

In [ ]:
for _, row in gdf.iterrows():
    iddpto = row["IDDPTO"]
    geometry = row["geometry"]
    for year in years:
        records.append({
            "IDDPTO": iddpto,
            "YEAR": year,
            "A": row.get(f"A_{year}", np.nan), # population
            "AMR": row.get(f"AMR_{year}", np.nan), # Asthma Mortality Rate
            "PM25": row.get(f"PM25_{year}", np.nan), # Mean PM2.5
            "PM10": row.get(f"PM10_{year}", np.nan), # Mean PM10
            "NBA": row.get(f"NBA_{year}", np.nan), # Normalized Burned Areas
            "PD": row.get(f"PD_{year}", np.nan), # Population Density
            "NAGRT": row.get(f"NAGRT_{year}", np.nan), # Normalized Agricultural and Livestock Transition areas
            "NNWVT": row.get(f"NNWVT_{year}", np.nan), # Normalized Natural Wooded Vegetation Transitions areas
            "NBUT": row.get(f"NBUT_{year}", np.nan), #  Normalized Built-up Transitions areas
            "geometry": geometry # Add geometry
            })

In [ ]:
# create new df from list and sort
panel_gdf = pd.DataFrame(records)

In [ ]:
# Sort and reset index
panel_gdf = panel_gdf.sort_values(by=["IDDPTO", "YEAR"]).reset_index(drop=True)

In [ ]:
# visualize the fisrt rows
init_notebook_mode(all_interactive=False)
panel_gdf.head()

In [ ]:
# visualize the number of rows and columsn of the data.frame
panel_gdf.shape

## 🧹 Data Quality

In this section we will inspect missing values per columns, check distribution of each feature and  verify temporal consistency of the dataset.

### Inspect missing values

In [ ]:
# drop gemetry column
df = panel_gdf.drop(columns=["geometry"])

In [ ]:
# check missing values
print(df.isna().sum())

In [ ]:
# check number of duplicated rows
print(df.duplicated().sum())

### Verify data consistency

In [ ]:
# Features to check (excluding ID, geometry)
features = ['A', 'AMR', 'PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT']

# Total zero counts per feature
zero_counts = (df[features] == 0).sum().sort_values(ascending=False)
print("Total Zero Values per Feature")
print(zero_counts)

In [ ]:
# Percentage of zero counts per feature
total_records = len(df)
zero_percentages = round((zero_counts / total_records) * 100, 2)
print("Percentage of Zero Values per Feature ")
print(zero_percentages.sort_values(ascending=False))

In [ ]:
# Heatmap visualization
# Calculate zero counts per feature per year
zero_counts_yearly = df.groupby('YEAR')[features].apply(lambda x: (x == 0).sum())

plt.figure(figsize=(12, 6))
sns.heatmap(zero_counts_yearly.T, annot=True, fmt="d", cmap="Reds")
plt.title("Zero Value Counts per Feature per Year")
plt.xlabel("Year")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 🔮 Predictor Variable Exploration

In this section we will explore the distribution, correlations, and spatio-tempral trends of the predictor variables.

### Visualization of Variable Distributions

Using the following code, we will generate histograms with kernel density estimation (KDE) curves to visualize the distribution of each variable and assess characteristics such as skewness, multimodality, and departures from normality. We will also create boxplots to examine the spread of the data and identify potential outliers.

In [ ]:
# Select only numeric variables for distribution check
numeric_vars = ['AMR', 'PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT']

In [ ]:
# define chart titles
titles = ['Asthma Mortality Rate (AMR)',
              'Particulate matter with a diameter\nof 2.5 micrometers or less (PM25)',
              'Particulate matter with a diameter\nof 2.5 micrometers or less (PM10)',
              'Normalized Burnes Areas (NBA)' ,
              'Population Density (PD)',
              'Normalized Agricultural and Livestock\nTransition areas (NAGRT)',
              'Normalized Natural Wooded Vegetation\nTransitions areas (NNWVT)',
              'Normalized Built-up Transitions areas (NBUT)'
]

In [ ]:
# define chart x labels
xlabels = ['Deaths/100,000 population',
              'Concentration in μg/m³',
              'Concentration in μg/m³',
              'km²/1,000' ,
              'People/km²',
              'km²/1,000',
              'km²/1,000',
              'km²/1,000'
]

In [ ]:
# Plot histograms + KDE
n_cols = 3
n_rows = (len(numeric_vars) + n_cols - 1) // n_cols

plt.figure(figsize=(16, n_rows * 4))

for i, (col, title, xlabel) in enumerate(
    zip(numeric_vars, titles, xlabels), 1
):
    plt.subplot(n_rows, n_cols, i)

    sns.histplot(
        df[col],
        kde=True,
        bins=30,
        color="steelblue"
    )

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")

plt.tight_layout()

# Save the chart with 300 dpi
# plt.savefig(
#     'pdt/asthma_mortality/data/images/figures/histograms_kde.png',
#     dpi=300,
#     bbox_inches='tight'
# )

plt.show()

In [ ]:
# show boxplots for skewed distributions
plt.figure(figsize=(16, n_rows * 3))
for i, col in enumerate(numeric_vars, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(x=df[col], color="orange")
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)

plt.tight_layout()
plt.show()

### Yearly trends of predictor variables

#### Ploting yearly trends for each predictor variable

In [ ]:
# visualize the data.frame
df.head()

In [ ]:
# copy dataframe
df_ts = df.copy()

In [ ]:
# Ensure YEAR is in datetime format and extract the year number
df_ts['YEAR_DT'] = pd.to_datetime(df_ts['YEAR'], format='%Y')
df_ts['year_num'] = df_ts['YEAR_DT'].dt.year

In [ ]:
df_ts.head()

In [ ]:
# Define predictor variables
predictors = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT']

In [ ]:
# Plot yearly trends for each predictor variable in separate subplots
n_predictors = len(predictors)  # Get the number of predictor variables
n_cols = 3  # Set the number of columns for subplots
n_rows = (n_predictors + n_cols - 1) // n_cols  # Calculate the number of rows needed for subplots

plt.figure(figsize=(15, n_rows * 5))  # Create a figure with a specified size

for i, var in enumerate(predictors, 1):  # Loop through each predictor variable
    plt.subplot(n_rows, n_cols, i)  # Create a subplot for the current variable
    yearly_mean = df_ts.groupby('year_num')[var].mean()  # Calculate yearly mean for the variable
    plt.plot(yearly_mean.index, yearly_mean.values, marker='o', linestyle='-')  # Plot the yearly trend
    plt.title(f"Yearly Trend of {var}")  # Set the title for the subplot
    plt.xlabel("Year")  # Label the x-axis
    plt.ylabel("Mean Value")  # Label the y-axis
    plt.grid(True)  # Enable grid for better visualization

plt.tight_layout()  # Adjust layout to prevent overlapping
plt.show()  # Display the plots

### Spatial patterns of predictor variables

In this section, we will create choropleth maps as faceted of predictor variables by year of study (2001-2022).  We will use [Pysal](https://pysal.org/)'s [mapclassify](https://pysal.org/mapclassify/index.html) library to determine the best classifier for the choropleth map by calculating the  best ACDM (mean Absolute Deviation Around the class Median). In Pysal, ACDM refers to the mean absolute deviation around the class median. It is a measure of a classifier's fit to the data, specifically by evaluating the average distance between each data point and the median value of the assigned class.

#### Particulate Matter < 2.5 μm ($PM_{2.5}$)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "PM25_"
pm25_columns = [col for col in gdf_cl.columns if col.startswith("PM25_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,pm25_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of $PM_{2.5}$


Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.2, 0.4),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": ["4.51-10.43", "10.43-13.56", "13.56-17.85", "17.85-24.82", "24.82-48.35"]  # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Annual mean concentration (µg/m³) by departments of $PM_{2.5}$ - 2001-2022', fontsize=14, y=1)

# row 0
maptma("PM25_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("PM25_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("PM25_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("PM25_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("PM25_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("PM25_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("PM25_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("PM25_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("PM25_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("PM25_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("PM25_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("PM25_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("PM25_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("PM25_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("PM25_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("PM25_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("PM25_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("PM25_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("PM25_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("PM25_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("PM25_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("PM25_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

#### Particulate Matter < 10 μm ($PM_{10}$)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "PM10_"
pm10_columns = [col for col in gdf_cl.columns if col.startswith("PM10_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,pm10_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of $PM_{10}$


Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.2, 0.35),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": ["99.00-114.15", "114.15- 119.6", " 119.6-126.9", "126.9-139.56", "139.56-167.59"]  # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Annual mean concentration (µg/m³) by departments of $PM_{10}$ - 2001-2022', fontsize=14, y=1)

# row 0
maptma("PM10_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("PM10_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("PM10_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("PM10_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("PM10_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("PM10_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("PM10_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("PM10_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("PM10_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("PM10_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("PM10_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("PM10_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("PM10_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("PM10_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("PM10_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("PM10_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("PM10_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("PM10_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("PM10_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("PM10_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("PM10_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("PM10_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

#### NBA (Normalized Burned Areas)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "NBA_"
nba_columns = [col for col in gdf_cl.columns if col.startswith("NBA_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,nba_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of NBA


Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

We will include a zero (0) value at the first position of the bin list, since in some departments and years no wildfires were detected by satellites.

In [ ]:
# insert 0 at 0 position
bins.insert(0, 0)
bins

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
# Extract intervals from bins
intervals = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]
intervals

In [ ]:
# add '0.00' at postion 0 to the list of intervals
intervals.insert(0, '0.00')
intervals

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.30, 0.4),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels":  intervals # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Normalized Burned Area (km$^2$) per 1000  km$^2$ by departments- 2001-2022', fontsize=14, y=1)

# row 0
maptma("NBA_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("NBA_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("NBA_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("NBA_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("NBA_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("NBA_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("NBA_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("NBA_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("NBA_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("NBA_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("NBA_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("NBA_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("NBA_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("NBA_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("NBA_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("NBA_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("NBA_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("NBA_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("NBA_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("NBA_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("NBA_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("NBA_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

#### Population density (PD)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "PD_"
pd_columns = [col for col in gdf_cl.columns if col.startswith("PD_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,pd_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of PD


We observe that the Fisher–Jenks and Head–Tail Breaks classifiers produced the lowest ACDM values. However, in this case, neither classifier provides a clear visualization of the spatial distribution of the PD variable. Therefore, we will use the Quantile classifier, which better highlights the spatial patterns present in the data.

In [ ]:
# Convert the bins to a list for further processing
bins = q4.bins.tolist()
bins

In [ ]:
# add 0.02 to bin for labeling
bins_label  = bins.copy()
bins_label.insert(0, 0.02)

# Extract intervals from bins
intervals = [f"{bins_label[i]:.2f}-{bins_label[i+1]:.2f}" for i in range(len(bins_label)-1)]
intervals

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.3, 0.35),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": intervals # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Population Density (km$^2$/hab) by departments - 2001-2022', fontsize=14, y=1)

# row 0
maptma("PD_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("PD_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("PD_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("PD_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("PD_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("PD_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("PD_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("PD_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("PD_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("PD_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("PD_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("PD_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("PD_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("PD_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("PD_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("PD_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("PD_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("PD_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("PD_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("PD_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("PD_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("PD_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

#### NAGRT (Normalized Agricultural and Livestock Transition areas)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "NAGRT_"
nagrt_columns = [col for col in gdf_cl.columns if col.startswith("NAGRT_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,nagrt_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of NAGRT


Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

In [ ]:
# insert 0 at 0 position
bins.insert(0, 0)
bins

In [ ]:
# Extract intervals from bins
intervals = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]
intervals

In [ ]:
# add '0.00' at postion 0 to the list of intervals
intervals.insert(0, '0.00')
intervals

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.25, 0.4),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": intervals  # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Normalized Agricultural and Livestock Transition areas (km$^2$) per 1000 km$^2$ by departments- 2001-2022', fontsize=14, y=1)

# row 0
maptma("NAGRT_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("NAGRT_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("NAGRT_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("NAGRT_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("NAGRT_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("NAGRT_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("NAGRT_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("NAGRT_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("NAGRT_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("NAGRT_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("NAGRT_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("NAGRT_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("NAGRT_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("NAGRT_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("NAGRT_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("NAGRT_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("NAGRT_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("NAGRT_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("NAGRT_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("NAGRT_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("NAGRT_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("NAGRT_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

#### NNWVT (Normalized Natural Wooded Vegetation Transitions areas)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "NNWVT_"
nnwvt_columns = [col for col in gdf_cl.columns if col.startswith("NNWVT_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,nnwvt_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of NNWVT

Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

In [ ]:
# insert 0 at 0 position
bins.insert(0, 0)
bins

In [ ]:
# Extract intervals from bins
intervals = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]
intervals

In [ ]:
# add '0.00' at postion 0 to the list of intervals
intervals.insert(0, '0.00')
intervals

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.25, 0.4),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": intervals  # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Normalized Natural Wooded Vegetation Transitions areas (km$^2$) per 1000 km$^2$ by departments- 2001-2022', fontsize=14, y=1)

# row 0
maptma("NNWVT_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("NNWVT_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("NNWVT_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("NNWVT_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("NNWVT_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("NNWVT_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("NNWVT_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("NAGRT_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("NNWVT_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("NNWVT_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("NNWVT_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("NNWVT_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("NNWVT_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("NNWVT_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("NNWVT_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("NNWVT_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("NNWVT_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("NNWVT_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("NNWVT_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("NNWVT_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("NNWVT_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("NNWVT_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

####NBUT (Normalized Built-up Transitions areas)

In [ ]:
# create a copy of  the geodataframe
gdf_cl = gdf.copy()

In [ ]:
# Get a list of column names that start with "NBUT_"
nbut_columns = [col for col in gdf_cl.columns if col.startswith("NBUT_")]

In [ ]:
# Select data to analize
selected_data = gdf_cl.loc[:,nbut_columns]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data, k=5)
fj5

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

##### Create choropleth maps of NBUT

Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

In [ ]:
# add 0.02 to bin for labeling
bins_label  = bins.copy()
bins_label.insert(0, 0.02)

# Extract intervals from bins
intervals = [f"{bins_label[i]:.2f}-{bins_label[i+1]:.2f}" for i in range(len(bins_label)-1)]
intervals

We will create a function to set a custom classification scheme based on FisherJenks

In [ ]:
def maptma(colname, title, row, col):
    # Create a custom classification using UserDefined
    classification = mapclassify.UserDefined(gdf_cl[colname], bins)

    classification.plot(
          gdf,  # GeoDataFrame containing the data to be plotted
          legend=True,  # Enable the legend for the plot
          legend_kwds={
                "fmt": "{:.0f}",  # Format the legend labels as integers
                # "loc": "upper right",  # Position the legend in the upper right corner
                "bbox_to_anchor": (1.2, 0.4),  # Adjust the legend's position
                "fontsize": 8,  # Set the font size of the legend
                "labels": intervals  # Use the custom legend labels
          },
          axis_on=False,  # Disable the axis display
          border_color='black',  # Set the border color of the plot
          cmap="viridis_r",  # Use the reversed Viridis colormap
          ax=axes[row, col]  # Specify the subplot to draw the plot on
     )

    # Set the title for the current axis
    axes[row, col].set_title(title)

In [ ]:
# Create a 5x5 grid of subplots with a figure size of 20x20
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
# Add a title to the entire figure with specific font size and position
plt.suptitle('Normalized Built-up Transitions areas by departments - 2001-2022', fontsize=14, y=1)

# row 0
maptma("NBUT_2001", "2001", 0, 0)  # Map data for 2001 in row 0, column 0
maptma("NBUT_2002", "2002", 0, 1)  # Map data for 2002 in row 0, column 1
maptma("NBUT_2003", "2003", 0, 2)  # Map data for 2003 in row 0, column 2
maptma("NBUT_2004", "2004", 0, 3)  # Map data for 2004 in row 0, column 3
maptma("NBUT_2005", "2005", 0, 4)  # Map data for 2005 in row 0, column 4

# row 1
maptma("NBUT_2006", "2006", 1, 0)  # Map data for 2006 in row 1, column 0
maptma("NBUT_2007", "2007", 1, 1)  # Map data for 2007 in row 1, column 1
maptma("NBUT_2008", "2008", 1, 2)  # Map data for 2008 in row 1, column 2
maptma("NBUT_2009", "2009", 1, 3)  # Map data for 2009 in row 1, column 3
maptma("NBUT_2010", "2010", 1, 4)  # Map data for 2010 in row 1, column 4

# row 2
maptma("NBUT_2011", "2011", 2, 0)  # Map data for 2011 in row 2, column 0
maptma("NBUT_2012", "2012", 2, 1)  # Map data for 2012 in row 2, column 1
maptma("NBUT_2013", "2013", 2, 2)  # Map data for 2013 in row 2, column 2
maptma("NBUT_2014", "2014", 2, 3)  # Map data for 2014 in row 2, column 3
maptma("NBUT_2015", "2015", 2, 4)  # Map data for 2015 in row 2, column 4

# row 3
maptma("NBUT_2016", "2016", 3, 0)  # Map data for 2016 in row 3, column 0
maptma("NBUT_2017", "2017", 3, 1)  # Map data for 2017 in row 3, column 1
maptma("NBUT_2018", "2018", 3, 2)  # Map data for 2018 in row 3, column 2
maptma("NBUT_2019", "2019", 3, 3)  # Map data for 2019 in row 3, column 3
maptma("NBUT_2020", "2020", 3, 4)  # Map data for 2020 in row 3, column 4

# row 4
maptma("NBUT_2021", "2021", 4, 0)  # Map data for 2021 in row 4, column 0
maptma("NBUT_2022", "2022", 4, 1)  # Map data for 2022 in row 4, column 1

axes[4,2].axis('off')  # Turn off axis for row 4, column 2
axes[4,3].axis('off')  # Turn off axis for row 4, column 3
axes[4,4].axis('off')  # Turn off axis for row 4, column 4

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show();  # Display the plot

### Correlation of predictor variables with target variable

#### Scatterplots and Linear Fits for Each Predictor

In [ ]:
predictors = ["PM25", "PM10", "NBA", "PD", "NAGRT", "NNWVT", "NBUT"]

plt.figure(figsize=(16, 12))

for i, var in enumerate(predictors, 1):
    plt.subplot(3, 3, i)
    sns.regplot(data=df_ts, x=var, y="AMR", scatter_kws={'alpha':0.3}, line_kws={"color":"red"})
    plt.title(f"AMR vs {var}")
    plt.xlabel(var)
    plt.ylabel("AMR")

plt.tight_layout()
plt.show()

#### Pearson & Spearman Correlations (AMR vs predictors)

In [ ]:
corr_results = {}

for var in predictors:
    pearson = df_ts["AMR"].corr(df_ts[var], method="pearson")
    spearman = df_ts["AMR"].corr(df_ts[var], method="spearman")
    corr_results[var] = {"Pearson": pearson, "Spearman": spearman}

corr_df = pd.DataFrame(corr_results).T
corr_df

#### Pretty heatmap of correlations

In [ ]:
plt.figure(figsize=(6, 6))
sns.heatmap(corr_df, annot=True, cmap="coolwarm", center=0, fmt=".3f")
plt.title("Correlation of AMR with Predictors")
plt.show()


## Interaction & Derived Features

### Lag effects

In [ ]:
# copy df as df_ts
df_ts = df.copy()

In [ ]:
# Define predictor variables and lags to test
predictors = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT']
lags = [1, 2]  # lag 1 and lag 2 years

In [ ]:
# Create lagged versions per department
df_lagged = df_ts.copy()

for var in predictors:
    for lag in lags:
        df_lagged[f'{var}_lag{lag}'] = df_lagged.groupby('IDDPTO')[var].shift(lag)

In [ ]:
# Compute correlations of CA with current and lagged predictors ---
corrs = {}

for var in predictors:
    corrs[var] = {
        'r_current': df_lagged['AMR'].corr(df_lagged[var]),
        'r_lag1': df_lagged['AMR'].corr(df_lagged[f'{var}_lag1']),
        'r_lag2': df_lagged['AMR'].corr(df_lagged[f'{var}_lag2'])
    }

corr_df = pd.DataFrame(corrs).T.round(3)
print("📈 Correlation of AMR with predictors and their lags:")
print(corr_df)

In [ ]:
# Visualize lag correlations as a heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(corr_df, annot=True, cmap="coolwarm", center=0)
plt.title("Lagged Correlation of Predictors with Asthma Mortality Rate (AMR)")
plt.ylabel("Predictor")
plt.xlabel("Lag type")
plt.tight_layout()
plt.show()

The lagged correlation analysis indicates that the associations between the predictors and the asthma mortality rate (AMR) are generally weak, with correlation coefficients remaining below 0.10 across all lag structures. $PM_{10}$ exhibits the strongest and most consistent positive relationship with AMR, reaching correlations of 0.099 for both the one-year and two-year lags, while NAGRT also shows a stable positive association (0.068–0.078). In contrast, $PM_{2.5}$ displays a weak negative correlation across all lags, and PD and NMWT are essentially uncorrelated with the outcome. NBA presents a modest contemporaneous correlation that diminishes and becomes negligible for lagged values, whereas NBUT maintains a small positive association. Overall, the results suggest that no single predictor has a strong linear relationship with AMR, supporting the use of multivariate and potentially non-linear modelling approaches to capture the complex determinants of asthma mortality.


### Correlation Matrix

In [ ]:
# define dataframe for corrrelation matrix
predictors = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT','NBUT']
X = panel_gdf[predictors].copy()

In [ ]:
#Correlation Matrix

corr_matrix = X.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title("Correlation Matrix of Predictors")
plt.show()

The correlation matrix indicates that the predictor variables exhibit generally weak pairwise associations, with all absolute correlation coefficients remaining below 0.35. The strongest positive relationship is observed between NBA and NNWVT (r = 0.34), followed by moderate correlations between $PM_{10}$ and NAGRT (r = 0.26), $PM_{10}$  and NBUT (r = 0.25), and NAGRT and NBUT (r = 0.24). The most pronounced negative association occurs between $PM_{2.5}$ and NAGRT (r = −0.26), while $PM_{2.5}$ also shows weak inverse relationships with $PM_{10}$ and the remaining predictors. Several variables, including NNWVT and NBUT, are essentially uncorrelated (r ≈ 0.00). Overall, the absence of strong pairwise correlations suggests that severe multicollinearity is unlikely to affect subsequent modelling.
